先训练再预测

## KNN（K-近邻算法）

> **核心思想**：物以类聚，人以群分。  
> 如果一个样本在特征空间中的 K 个最相似（即特征空间中距离最近）的样本中的大多数属于某一个类别，则该样本也属于这个类别。

**分类与回归的统一框架：**
- **分类任务**：投票机制（多数表决）
- **回归任务**：均值机制（取平均值）

---

### K 值的影响（超参数选择）

| K 值 | 影响 | 后果 |
| :--- | :--- | :--- |
| **K 值过小** | 模型复杂，容易受异常点干扰 | **过拟合**（对噪声敏感） |
| **K 值过大** | 模型简单，受整体样本分布影响 | **欠拟合**（忽略局部特征） |

分类是投票 回归是均值
相同点：有监督学习（有特征，有标签）
不同点：分类（标签不连续）、回归（标签连续）

### 1. 分类任务（标签离散）

1. 计算未知样本到**每一个**训练样本的距离
2. 将训练样本按距离大小**升序排列**
3. 取出距离最近的 **K 个**训练样本
4. **多数表决（投票）**：统计 K 个样本中哪个类别出现的次数最多
5. 将未知样本归属到**出现次数最多的类别**

---

### 2. 回归任务（标签连续）

1. 计算未知样本到**每一个**训练样本的距离
2. 将训练样本按距离大小**升序排列**
3. 取出距离最近的 **K 个**训练样本
4. **计算平均值**：将这 K 个样本的目标值取均值
5. 将该均值作为未知样本的预测值


---



## 常用的距离度量方式

### 1. 欧氏距离（Euclidean Distance） 

**直觉**：两点之间的直线距离（最常见的距离），对应维度差值平方和，开平方根。

**公式（二维）：**
$$
d = \sqrt{(x_1 - y_1)^2 + (x_2 - y_2)^2}
$$

**公式（n 维）：**
$$
d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}
$$

### 2. 曼哈顿距离（Manhattan Distance）

**直觉**：只能沿坐标轴方向移动的距离，如同在曼哈顿街区从 A 点走到 B 点。
         也称"城市街区距离"，对应维度差值的绝对值，求和。

**公式：**
$$
d(\mathbf{x}, \mathbf{y}) = \sum_{i=1}^{n} |x_i - y_i|
$$

### 3. 切比雪夫距离（Chebyshev Distance）

**直觉**：可以斜着走，国际象棋中**国王走一步**覆盖的范围（可以斜着走），对应维度差值的绝对值，求最大值。

**公式：**
$$
d(\mathbf{x}, \mathbf{y}) = \max_{i} |x_i - y_i|
$$

## 特征值预处理

> **为什么需要预处理？**  
> 特征的单位或大小范围相差较大时（如 "年龄" 0\~100 和 "年收入" 0\~100000），数值大的特征会对距离计算产生主导影响，导致模型偏向于该特征。


### 1. 归一化（Min-Max Scaling）

**原理**：将原始数据线性变换到 `[min, max]` 区间，默认映射到 `[0, 1]`。

**公式：**
$$
X' = \frac{x - \min(x)}{\max(x) - \min(x)}$$
$$
X'' = X' \times (\text{new\_max} - \text{new\_min}) + \text{new\_min}
$$

**特点**：
- 容易受到最大值和最小值的影响（**鲁棒性较差**）
- 适用于**精确小数据场景**

### 2. 标准化（Standardization / Z-Score Normalization）

**原理**：将原始数据转换为均值为 0，标准差为 1 的标准正态分布。

**公式：**
$$
X' = \frac{x - \text{mean}}{\sigma}
$$

> - `mean`：该特征的均值  
> - `σ`（sigma）：该特征的标准差

**特点**：
- 少量异常点对于平均值影响不大
- 适用于**现代嘈杂大数据场景**

## 交叉验证（Cross-Validation）

> **目的**：将数据集划分为训练集和验证集，多次评估模型，得到更准确可信的模型评分，从而选出最优超参数 K。

### 交叉验证原理（以 n 折交叉验证为例）

1. **第 1 次**：将第 1 份数据作为验证集，其余 `n-1` 份作为训练集
2. **第 2 次**：将第 2 份数据作为验证集，其余 `n-1` 份作为训练集
3. ……以此类推，共训练 `n` 次，评估 `n` 次
4. 计算 `n` 次评估结果的**平均值**，作为该超参数 K 的交叉验证得分
5. 选出**得分最高的 K 值**（如 `K = i` 时得分最好）
6. 使用**全部训练集**（原训练集 + 验证集）对 `K = i` 的模型**重新训练**
7. 最后使用**独立的测试集**对最终模型进行评估


## 网格搜索（Grid Search）

> **自动化调参工具**：网格搜索是一种**超参数优化**方法，它会遍历你指定的所有候选超参数组合，并对每一组组合执行一次**交叉验证评估**，最终自动选出平均分最高的那一组参数。

### 网格搜索的核心逻辑

- **外层循环**：遍历所有超参数组合（如 `n_neighbors` = [3,5,7]，`weights` = ['uniform','distance']，共 3×2 = 6 种组合）
- **内层循环**：对每一组参数，执行一次完整的交叉验证（如 `cv=5`），得到该组参数的平均得分
- **输出结果**：返回交叉验证得分最高的参数组合（`best_params_`）及其得分（`best_score_`）

### 与交叉验证的关系（一句话概括）

> **交叉验证是网格搜索内部的“评分引擎”；网格搜索是交叉验证的“调度者”。**

- **交叉验证**：告诉你“这组参数在数据上表现如何”（打一个分数）。
- **网格搜索**：不断尝试新的参数组合，反复调用交叉验证，最后挑出最佳组合。

### 为什么需要网格搜索？

- **自动化**：避免手动修改代码、重跑、记录分数的繁琐流程。
- **系统化**：确保所有候选参数都被公平评估，不会遗漏。
- **效率**：一次定义好参数网格，程序自动完成所有训练和评估。


原始数据集
    │
    ├── 训练集（用于调参）
    │       │
    │       └── 网格搜索（外层遍历参数组合）
    │               │
    │               ├── 组合1 (K=3, weights='uniform')
    │               │       └── 交叉验证 (cv=5) → 得分1
    │               ├── 组合2 (K=3, weights='distance')
    │               │       └── 交叉验证 (cv=5) → 得分2
    │               ├── 组合3 ...（循环所有组合）
    │               │
    │               └── 选出平均分最高的参数组合（best_params_）
    │
    ├── 使用全部训练集重新训练模型（最佳参数）
    │
    └── 测试集 → 评估最终模型的泛化能力